# 09C: Internal-lock cached prediction

**Notebook file:** `09c-toysad8k-protected-internal-lock-inference.ipynb`

This prepared notebook uses the frozen **automatic winner consensus** and **correspondence-assisted foreground patch-bank strongest-two fusion**. It preserves all 400 original `internal_lock` scenes. Two, three and five views are core settings; one view is diagnostic. The nineteen schedules are fixed before evaluator access.

This is a current-method adapter, not the old P1-only body. Preparation and synthetic tests do not establish a completed protected run. The source snapshot, current runtime freeze, the declared asset-exposure policy and exact native cache/evaluator resources must pass the executable preflight.

## Resolve the complete source snapshot and explicit stage configuration

Mount the prepared source package or set `VPI_CODE_ROOT`. The standard-library bootstrap verifies the source manifest and exact Python/dependencies before scientific imports. Set `VPI_NEXT_CONFIG` or mount the unambiguous stage configuration. Nested Kaggle inputs resolve to their sealed package roots. Do not attach a full old evaluator notebook output to prediction stages. This notebook contains no credentials, model download or inference command.

In [ ]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = None
NOTEBOOK_RELATIVE = 'notebooks/protected-evaluation/09c-toysad8k-protected-internal-lock-inference.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    BOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)
    CODE_PROVENANCE = BOOTSTRAP['provenance']
    ROOT = CODE_ROOT
    # Scientific imports are allowed only after the exact runtime/source preflight.
    from IPython.display import display, Image
    import pandas as pd
    display({'source': CODE_PROVENANCE, 'runtime': BOOTSTRAP['runtime'],
             'preparation_only': 'Verified source/runtime does not establish protected-resource readiness'})
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _failure = {'status':'bootstrap_failed','error_type':type(_bootstrap_error).__name__,'scientific_run_started':False}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure,sort_keys=True)+'\n')
    os.replace(_temporary,_failure_root/'bootstrap-failure.json')
    raise

In [ ]:
cfg=_runtime.resolve_config('09c',ROOT)
from vpi_next_evaluation.contracts import current_method_contract
from vpi_next_evaluation.original import readiness, predict_current, evaluate_predictions, release_internal, release_calibration
from vpi_remaining_evaluation.gates import PrerequisiteBlocked
from vpi_object_bank.storage import read_json, verify_seal, atomic_json
from current_stage_execution import run_stage
display(current_method_contract())

## Verify native resources, exposure and the current runtime freeze

The method freeze is produced from the completed 09E decision plus the current application's source and runtime versions. The old 08F/P1 freeze cannot pass. Before this sequence, inventory compatible outputs for the remaining 900 development scenes, verify any defensible confirmation cohort, and prepare the calibration cache/resource bridge. CPU processing reuses saved scores and patches. The separate current input-preparation adapter extracts missing DINO patches with a pinned CPU FP32 neural runtime; genuinely missing LocateAnything or SAM regions use the separate current-native-region-inference notebook.

An absent or invalid asset-exposure ledger stops execution. The explicitly exposure-qualified policy retains unknown access history and forbids independent or untouched claims. Correct scene counts without the canonical opaque-scene membership hash cannot pass. Native cache preparation and reviewed scene-to-asset attribution are real prerequisites; no readiness flag replaces them.

In [ ]:
def check_stage_resources():
    preflight=readiness('09c',cfg,ROOT)
    if preflight['status']!='prerequisites_present':
        raise PrerequisiteBlocked('CURRENT_STAGE_BLOCKED',preflight['detail'])
    return preflight
preflight=run_stage('09c-preflight',cfg,check_stage_resources,CODE_PROVENANCE)
display(preflight)

## Produce both fixed decisions from selected-view cached evidence

This CPU stage receives no anomaly labels. It preserves every valid observation for assisted fusion and records the automatic winner-consensus retention separately. Unknown assisted IDs stay unknown; each ranking carries support and per-view component scores. Same-view duplicate assignments and unselected observations are rejected. Empty views remain explicit.

Scene checkpoints bind cache, method, exposure, source and schedule hashes. Interrupted units remain recovery evidence. Only a complete original-role denominator can create the final prediction lock.

In [ ]:
receipt=run_stage('09c',cfg,lambda:predict_current('09c',cfg,ROOT),CODE_PROVENANCE)
display(receipt)
verify_seal(Path(cfg['output_root']),name='prediction_lock.json')

## Inspect sealed prediction-only scores and hand off

Per-scene predictions and `per_view_scores.csv` are private. They include one combined ranking for each fixed setting. Rich RGB, masks, boxes, score matrices and projected coarse evidence are rendered after the evaluator boundary by the report adapter; missing cached media is reported explicitly. No evaluator should be mounted here.

In [ ]:
display({'prediction_lock':'verified','private_score_exports':'scenes/<opaque ID>/per_view_scores.csv','evaluation_opened':False,'next':'release this model-only prediction resource for its fixed evaluator'})

## Recovery and bounded publication

Scientific results are sealed before plotting or tracking. `scripts/publish_next_stage.py` publishes the explicit aggregate table/figure allowlist in a 180-second child process. Run it separately with `--stage`, `--evaluation-root`, `--report-root` and `--receipt` after verification. Keep private RGB/masks, object scores, raw IDs and evaluator records out of ClearML. A tracking failure leaves a pending receipt and never reruns prediction or inference. Kaggle interruption recovery requires exported checkpoint files; local checkpoint existence alone does not survive a lost session.

In [ ]:
tracking={'status':'pending_separate_publication','private_artifacts_allowed':False,'scientific_results_preserved':True}
atomic_json(Path(cfg['output_root']).parent/'tracking'/'09c-pending.json',tracking)
display(tracking)